### Runnables 구조 검토

In [1]:
!uv --version

uv 0.12.17 (635500036 2026-09-18 x86_64-pc-windows-msvc)


In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
# !uv add faiss-cpu tiktoken

In [ ]:
# !uv add grandalf

In [5]:
from langchain_classic.prompts import ChatPromptTemplate
from langchain_classic.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

vectorstore = FAISS.from_texts(
    ["Teddy is an AI engineer who loves programming!"],
    embedding=OpenAIEmbeddings(),
)

retriever = vectorstore.as_retriever()

template = """Answer the question based only on the following context: {context}

Question: {question}"""

prompt = ChatPromptTemplate.from_template(template)

model = ChatOpenAI(model="gpt-4o-mini")

chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [6]:
chain.get_graph().nodes

{'42cb3c8d8cb947e29b94dcbbf4035126': Node(id='42cb3c8d8cb947e29b94dcbbf4035126', name='Parallel<context,question>Input', data=<class 'langchain_core.runnables.base.RunnableParallel<context,question>Input'>, metadata=None),
 'c1f9db85408c4721a6eee5bc6a2ed45d': Node(id='c1f9db85408c4721a6eee5bc6a2ed45d', name='Parallel<context,question>Output', data=<class 'langchain_core.utils.pydantic.RunnableParallel<context,question>Output'>, metadata=None),
 'd7d65c6c80424101892584ddf71be897': Node(id='d7d65c6c80424101892584ddf71be897', name='VectorStoreRetriever', data=VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001A96DDE60C0>, search_kwargs={}), metadata=None),
 'b6ced2e270884d70803b5596b16769b7': Node(id='b6ced2e270884d70803b5596b16769b7', name='Passthrough', data=RunnablePassthrough(), metadata=None),
 '1cedc22678eb47c5b69076795c1d2694': Node(id='1cedc22678eb47c5b69076795c1d2694', name='ChatPromptTemplate', dat

In [7]:
chain.get_graph().edges

[Edge(source='dd17d46118124191bab07e608299fbbe', target='a7273c67efaa4d1e95b99273bef72007', data=None, conditional=False),
 Edge(source='a7273c67efaa4d1e95b99273bef72007', target='899cedc68dca44e086ed14e674a9f75f', data=None, conditional=False),
 Edge(source='dd17d46118124191bab07e608299fbbe', target='628b0df223df4c128ab7d3c296a849e2', data=None, conditional=False),
 Edge(source='628b0df223df4c128ab7d3c296a849e2', target='899cedc68dca44e086ed14e674a9f75f', data=None, conditional=False),
 Edge(source='899cedc68dca44e086ed14e674a9f75f', target='b5c0a197605a477e8fc73cba2dc42674', data=None, conditional=False),
 Edge(source='b5c0a197605a477e8fc73cba2dc42674', target='3976908a5c7043e9a11f80d0ac740561', data=None, conditional=False),
 Edge(source='c28ccb7ae96b413fbeaeeca164f7daca', target='f44980bd64954617b7753a8c8d3953c6', data=None, conditional=False),
 Edge(source='3976908a5c7043e9a11f80d0ac740561', target='c28ccb7ae96b413fbeaeeca164f7daca', data=None, conditional=False)]

In [8]:
chain.get_graph().print_ascii()

           +---------------------------------+         
           | Parallel<context,question>Input |         
           +---------------------------------+         
                    **               **                
                 ***                   ***             
               **                         **           
+----------------------+              +-------------+  
| VectorStoreRetriever |              | Passthrough |  
+----------------------+              +-------------+  
                    **               **                
                      ***         ***                  
                         **     **                     
           +----------------------------------+        
           | Parallel<context,question>Output |        
           +----------------------------------+        
                             *                         
                             *                         
                             *                  

In [9]:
chain.get_prompts()

[ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context: {context}\n\nQuestion: {question}'), additional_kwargs={})])]